In [ ]:
# ==============================================================================
# 3_model_training.ipynb
# ==============================================================================
import time
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, GBTClassifier, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("\n>>> STEP 4: Training 4 Big Data Models (Rubric: 3+ Algorithms)...")

# 1. Define Algorithms
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)
svm = LinearSVC(labelCol="label", featuresCol="features", maxIter=10)
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=10, seed=42)

# 2. Setup Evaluators
eval_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

# 3. Hyperparameter Tuning
paramGrid_rf = ParamGridBuilder().addGrid(rf.numTrees, [10, 20]).build()
cv_rf = CrossValidator(estimator=rf, estimatorParamMaps=paramGrid_rf, evaluator=eval_auc, numFolds=2, parallelism=2)

# 4. Model List
models = [
    ("Random Forest (CV)", cv_rf),
    ("Gradient Boosted", gbt),
    ("Logistic Reg", lr),
    ("Linear SVM", svm)
]

print(f"\n{'Algorithm':<20} | {'AUC':<7} | {'Acc':<7}")
print("-" * 40)

best_higgs_model_vraj = None
best_predictions = None

# 5. Training Loop
for name, algo in models:
    try:
        start_time = time.time()
        model = algo.fit(train_data)
        preds = model.transform(test_data)

        auc = eval_auc.evaluate(preds)
        acc = eval_acc.evaluate(preds)

        print(f"{name:<20} | {auc:.4f} | {acc:.4f} | {(time.time() - start_time):.1f}s")

        if name == "Gradient Boosted":
            best_higgs_model_vraj = model
            best_predictions = preds

    except Exception as e:
        print(f"{name:<20} | FAILED: {str(e)[:50]}")

# 6. SERIALIZATION: Save the GBT Model
if best_higgs_model_vraj:
    model_path = "/content/drive/MyDrive/Bigdata/higgs_gbt_model"
    best_higgs_model_vraj.write().overwrite().save(model_path)
    print(f"\nSUCCESS: GBT Model serialized and saved to: {model_path}")